# Cleaning Data — Titanic Dataset

**Track:** Data Analytics (Level 1) | **Internship:** Oasis Infobyte SIP | **Author:** Shreya Kadam

**Objective:** Take a deliberately messy dataset and systematically clean it into an analysis-ready dataset, documenting every decision.

**Dataset:** Titanic Dataset (built into seaborn)

**Tech Stack:** Python, pandas, numpy, seaborn, Jupyter Notebook

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

df = pd.read_csv("titanic.csv")
print(df.shape)
df.head()

(1309, 28)


,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,2urvived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


## Data Quality Report (Before Cleaning)

In [7]:
print("--- Null counts per column ---")
print(df.isnull().sum())

print("\n--- Duplicate rows ---")
print(df.duplicated().sum())

print("\n--- Data types ---")
print(df.dtypes)

print("\n--- Columns ---")
print(df.columns.tolist())

--- Null counts per column ---
Passengerid    0
Age            0
Fare           0
Sex            0
sibsp          0
zero           0
zero.1         0
zero.2         0
zero.3         0
zero.4         0
zero.5         0
zero.6         0
Parch          0
zero.7         0
zero.8         0
zero.9         0
zero.10        0
zero.11        0
zero.12        0
zero.13        0
zero.14        0
Pclass         0
zero.15        0
zero.16        0
Embarked       2
zero.17        0
zero.18        0
2urvived       0
dtype: int64

--- Duplicate rows ---
0

--- Data types ---
Passengerid      int64
Age            float64
Fare           float64
Sex              int64
sibsp            int64
zero             int64
zero.1           int64
zero.2           int64
zero.3           int64
zero.4           int64
zero.5           int64
zero.6           int64
Parch            int64
zero.7           int64
zero.8           int64
zero.9           int64
zero.10          int64
zero.11          int64
zero.12          int

In [8]:
before_stats = {
    "null_count": df.isnull().sum().sum(),
    "duplicate_count": df.duplicated().sum(),
    "row_count": df.shape[0],
    "column_count": df.shape[1]
}
before_stats

{'null_count': np.int64(2),
 'duplicate_count': np.int64(0),
 'row_count': 1309,
 'column_count': 28}

## Removing Junk Columns

This dataset contains 19 columns named `zero`, `zero.1` through `zero.18` — these are placeholder/junk columns with no real information (likely artifacts from how this dataset was exported). They add no analytical value, so we drop them all.

In [9]:
junk_columns = [col for col in df.columns if col.startswith("zero")]
print(f"Dropping {len(junk_columns)} junk columns:", junk_columns)

df = df.drop(columns=junk_columns)
print("\nRemaining columns:", df.columns.tolist())

Dropping 19 junk columns: ['zero', 'zero.1', 'zero.2', 'zero.3', 'zero.4', 'zero.5', 'zero.6', 'zero.7', 'zero.8', 'zero.9', 'zero.10', 'zero.11', 'zero.12', 'zero.13', 'zero.14', 'zero.15', 'zero.16', 'zero.17', 'zero.18']

Remaining columns: ['Passengerid', 'Age', 'Fare', 'Sex', 'sibsp', 'Parch', 'Pclass', 'Embarked', '2urvived']


## Fixing Column Names

The target column is named `2urvived` — clearly a typo for `Survived` (likely from a corrupted export). We rename it for clarity. We also standardize all column names to a consistent case.

In [12]:
df = df.rename(columns={
    "2urvived": "Survived",
    "Passengerid": "PassengerId",
    "sibsp": "SibSp"
})
print(df.columns.tolist())
df.head()

['PassengerId', 'Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Pclass', 'Embarked', 'Survived']


,PassengerId,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived
0,1,22.0,7.2500,0,1,0,3,2.0,0
1,2,38.0,71.2833,1,1,0,1,0.0,1
2,3,26.0,7.9250,1,0,0,3,2.0,1
3,4,35.0,53.1000,1,1,0,1,2.0,1
4,5,35.0,8.0500,0,0,0,3,2.0,0


## Missing Data Handling

In [13]:
print(df.isnull().sum())

PassengerId    0
Age            0
Fare           0
Sex            0
SibSp          0
Parch          0
Pclass         0
Embarked       2
Survived       0
dtype: int64


In [14]:
# Fill Age with median (robust to outliers)
if df["Age"].isnull().sum() > 0:
    df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Fare with median if any missing
if df["Fare"].isnull().sum() > 0:
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Fill Embarked with mode if any missing
if df["Embarked"].isnull().sum() > 0:
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("Remaining nulls:\n", df.isnull().sum())

Remaining nulls:
 PassengerId    0
Age            0
Fare           0
Sex            0
SibSp          0
Parch          0
Pclass         0
Embarked       0
Survived       0
dtype: int64


## Duplicate Removal

In [15]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicate rows found: {duplicates_before}")
print(f"Remaining rows: {df.shape[0]}")

Duplicate rows found: 0
Remaining rows: 1309


## Standardisation

Checking the `Sex` and `Embarked` columns for inconsistent formatting (e.g. mixed case, extra spaces).

In [16]:
print("Unique Sex values:", df["Sex"].unique())
print("Unique Embarked values:", df["Embarked"].unique())
print("Unique Pclass values:", df["Pclass"].unique())

Unique Sex values: [0 1]
Unique Embarked values: [2. 0. 1.]
Unique Pclass values: [3 1 2]


## Outlier Detection (Fare)

In [17]:
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["Fare"] < lower_bound) | (df["Fare"] > upper_bound)]
print(f"Number of Fare outliers: {len(outliers)}")
print(f"Valid range: {lower_bound:.2f} to {upper_bound:.2f}")

Number of Fare outliers: 171
Valid range: -27.17 to 66.34


## Outlier Decision

High fares genuinely correspond to first-class passengers — this is real data, not an error. We retain these values rather than removing them, since doing so would distort the dataset's representation of actual passenger fare distribution.

## Data Type Correction

In [18]:
print("Before:\n", df.dtypes)

df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].astype("category")
df["Survived"] = df["Survived"].astype(int)
df["Pclass"] = df["Pclass"].astype(int)

print("\nAfter:\n", df.dtypes)

Before:
 PassengerId      int64
Age            float64
Fare           float64
Sex              int64
SibSp            int64
Parch            int64
Pclass           int64
Embarked       float64
Survived         int64
dtype: object

After:
 PassengerId       int64
Age             float64
Fare            float64
Sex            category
SibSp             int64
Parch             int64
Pclass            int64
Embarked       category
Survived          int64
dtype: object


## Before vs. After Summary

In [19]:
after_stats = {
    "null_count": df.isnull().sum().sum(),
    "duplicate_count": df.duplicated().sum(),
    "row_count": df.shape[0],
    "column_count": df.shape[1]
}

summary = pd.DataFrame({
    "Metric": ["Null Values", "Duplicate Rows", "Row Count", "Column Count"],
    "Before": [before_stats["null_count"], before_stats["duplicate_count"], before_stats["row_count"], before_stats["column_count"]],
    "After": [after_stats["null_count"], after_stats["duplicate_count"], after_stats["row_count"], after_stats["column_count"]]
})
summary

,Metric,Before,After
0,Null Values,2,0
1,Duplicate Rows,0,0
2,Row Count,1309,1309
3,Column Count,28,9


In [20]:
df.to_csv("titanic_cleaned.csv", index=False)
print("Cleaned dataset saved as titanic_cleaned.csv")
df.head()

Cleaned dataset saved as titanic_cleaned.csv


,PassengerId,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived
0,1,22.0,7.2500,0,1,0,3,2.0,0
1,2,38.0,71.2833,1,1,0,1,0.0,1
2,3,26.0,7.9250,1,0,0,3,2.0,1
3,4,35.0,53.1000,1,1,0,1,2.0,1
4,5,35.0,8.0500,0,0,0,3,2.0,0
